## software
pytorch

### autograd

In [ ]:
import torch 

N, D_in, H, D_out = 64,1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)
w1 = torch.randn(D_in, H, requires_grad=True) # 自动构建计算图存储该变量导数
w2 = torch.randn(H, D_out, requires_grad=True)

learning_rate = 1e-6
for t in range(50):
    y_pred = x.mm(w1).clamp(min=0).mm(w2)
    loss = (y_pred - y).pow(2).sum()

    loss.backward() # 反向传播算梯度，只会算有grad = True的变量

    with torch.no_grad(): # 告诉pytorch不要build graphs for these oerations
        w1 -= learning_rate * w1.grad   
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad

        w1.grad.zero_() # 结束一次iteration之后记得把grad归零，不然第二次iteration会累加
        w2.grad.zero_()

### new functions
#### 1. can define new operations using python functions

but they will add to the graph(giant graph) each time they are called


In [ ]:
def sigmoid(x):
    return 1/(1+np.exp(-x))


#### 2. Difine new autograd operatiors

siple node in computation graph

In [ ]:
class Sigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return 1 / (1 + torch.exp(-input))

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        return grad_output * torch.exp(-input) / (1 + torch.exp(-input)) ** 2
    
def sigmoid(x):
    return Sigmoid.apply(x)

### nn & optim

In [ ]:
import torch
N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

model  = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) # 选择优化器

for t in range(500):
    y_pred = model(x)
    loss = torch.nn.functional.mse_loss(y_pred, y) # 选择损失函数

    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

### 